# AI Workshop - MLCon Berlin 2025

## What this notebook covers

This workshop walks you through building an **Agentic RAG (Retrieval Augmented Generation)** system:

1. **Setup & LLM Clients** - Configure OpenAI and Groq clients
2. **Basic LLM Calls** - Test chat completions with both providers
3. **Document Loading** - Load FAQ documents for our knowledge base
4. **Search Index** - Build a searchable index using minsearch
5. **Tool Definition** - Define a search tool for the LLM to use
6. **Manual RAG** - Build context-augmented prompts manually
7. **Agentic RAG** - Let the LLM decide when to search (tool calling)
8. **Agentic Loop** - Automate the tool-calling cycle

## Next steps (to migrate from reference notebook)
- Add `make_call` helper function
- Add `developer_prompt` for better agent behavior  
- Add automated agentic loop with `while True`
- Explore `toyaikit` for chat interfaces
- OpenAI Agents SDK integration
- Pydantic AI integration
- MCP (Model Context Protocol) server

---

## 1. Setup & Dependencies

In [ ]:
import os
import json
import requests
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

openai_client = OpenAI()

## 2. LLM Clients Setup

In [ ]:
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Tell me a short Christmas story"}
    ]
)

print(response.choices[0].message.content)

In [ ]:
# Groq client (OpenAI-compatible API)
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

# Test Groq client
response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Tell me a short HannukaH story"}
    ]
)

print(response.choices[0].message.content)

In [ ]:
# Choose which client to use for the rest of the notebook
client = groq_client  # or groq_client

In [ ]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="Write a short bedtime story about a unicorn."
)

print(response.output_text)

In [ ]:
response = client.responses.create(
    model="openai/gpt-oss-20b",
    input="Write a short bedtime story about a unicorn."
)

print(response.output_text)

In [ ]:
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()
print(documents_raw)

## 3. Document Loading & Indexing

In [ ]:
documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [ ]:
documents[12]

In [ ]:
from minsearch import AppendableIndex

In [ ]:
index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
    )

    return results

## 4. Search Function & Tool Definition

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
question = 'I just discovered the course. Can I join it now?'

In [ ]:
result = search(question)
print(result)

In [ ]:
prompt = f"""
Answer the question from the student using the provided context

<QUESTION>{question}</QUESTION>

<CONTEXT>{json.dumps(result)}</CONTEXT>
"""

## 5. Manual RAG - Building Context Prompts

In [ ]:
# agentic RAG

chat_messages = [
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)

## 6. Agentic RAG - Tool Calling

In [ ]:
print(response)

In [ ]:
tool_call = response.output[0]
tool_call

In [ ]:
chat_messages.append(tool_call)

In [ ]:
search_result = search(query="Can I join the course now?")

In [ ]:
result_json = json.dumps(search_result, indent=2)

chat_messages.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": result_json,
})

In [ ]:
chat_messages

In [ ]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)

In [ ]:
response.output_text

In [ ]:
chat_messages.append(
    {"role": "user", "content": "but are you sure I can get my certificate?"}
)

In [ ]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)
response.output_text

## 7. Automated Agentic Loop

Now let's automate the tool-calling cycle with a helper function and a loop.

In [ ]:
def make_call(call):
    """Execute a tool call and return the result in the expected format."""
    args = json.loads(call.arguments)
    f_name = call.name
    f = globals()[f_name]
    result = f(**args)
    result_json = json.dumps(result, indent=2)
    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

If you want to look up the answer, explain why before making the call. Use as many 
keywords from the user question as possible when making first requests.

Make multiple searches. Try to expand your search by using new keywords based on the results you
get from the search.

At the end, make a clarifying question based on what you presented and ask if there are 
other areas that the user wants to explore.
""".strip()

In [ ]:
question = "I just discovered the course, can I join it now?"

chat_messages = [
    {"role": "developer", "content": developer_prompt},
    {"role": "user", "content": question}
]

In [ ]:
# Automated agentic loop
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=chat_messages,
        tools=[search_tool]
    )
    
    chat_messages.extend(response.output)

    has_function_calls = False
    
    for entry in response.output:
        if entry.type == 'message':
            print(entry.content[0].text)
        if entry.type == 'function_call':
            print(f"[Tool call: {entry.name}({entry.arguments})]")
            result = make_call(entry)
            chat_messages.append(result)
            has_function_calls = True

    if not has_function_calls:
        break

## 8. Using toyaikit for Chat Interfaces

The `toyaikit` library provides abstractions for building interactive chat agents.

In [ ]:
# Install toyaikit if needed
# !uv pip install toyaikit

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.runners import DisplayingRunnerCallback

In [ ]:
# Register tools with toyaikit
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [ ]:
# Create chat interface and runner
chat_interface = IPythonChatInterface()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    llm_client=OpenAIClient()
)

In [ ]:
# Run interactive chat (type 'stop' to end)
callback = DisplayingRunnerCallback(chat_interface)
messages = runner.loop(prompt='how do I install kafka', callback=callback)

In [ ]:
# Continue conversation with previous context
new_messages = runner.loop(
    prompt='I want to use docker',
    previous_messages=messages.all_messages,
    callback=callback,
)

In [ ]:
# Or run fully interactive session
# messages = runner.run();

## 9. OpenAI Agents SDK

The OpenAI Agents SDK provides a higher-level abstraction for building agents.

In [ ]:
# Install if needed: !uv pip install openai-agents

from agents import Agent, function_tool
from toyaikit.tools import wrap_instance_methods

In [ ]:
# Create a class to hold our search tools
from typing import List, Dict, Any

class SearchTools:

    def __init__(self, index):
        self.index = index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the FAQ database for entries matching the given query.
    
        Args:
            query (str): Search query text to look up in the course FAQ.
    
        Returns:
            List[Dict[str, Any]]: A list of search result entries, each containing relevant metadata.
        """
        boost = {'question': 3.0, 'section': 0.5}
    
        results = self.index.search(
            query=query,
            filter_dict={'course': 'data-engineering-zoomcamp'},
            boost_dict=boost,
            num_results=5,
        )
    
        return results

    def add_entry(self, question: str, answer: str) -> None:
        """
        Add a new entry to the FAQ database.
    
        Args:
            question (str): The question to be added to the FAQ database.
            answer (str): The corresponding answer to the question.
        """
        doc = {
            'question': question,
            'text': answer,
            'section': 'user added',
            'course': 'data-engineering-zoomcamp'
        }
        self.index.append(doc)

search_tools = SearchTools(index)

In [ ]:
# Wrap methods as function tools for the agent
tools = wrap_instance_methods(function_tool, search_tools)

In [ ]:
# Create the agent
agent = Agent(
    name="faq_agent",
    instructions=developer_prompt,
    tools=tools,
    model='gpt-4o-mini'
)

In [ ]:
# Create runner with toyaikit
from toyaikit.chat.runners import OpenAIAgentsSDKRunner

runner = OpenAIAgentsSDKRunner(
    chat_interface=chat_interface,
    agent=agent
)

In [ ]:
# Run the agent (type 'stop' to end)
await runner.run();

## 10. Pydantic AI

Pydantic AI provides another framework for building agents with type safety.

In [ ]:
# Install if needed: !uv pip install pydantic-ai

from pydantic_ai import Agent

In [ ]:
# Pydantic AI can use methods directly as tools
pydantic_tools = [
    search_tools.search,
    search_tools.add_entry
]

pydantic_tools

In [ ]:
# Create Pydantic AI agent
pydantic_agent = Agent(
    name="faq_agent",
    instructions=developer_prompt,
    tools=pydantic_tools,
    model='openai:gpt-4o-mini'  # or 'anthropic:claude-3-7-sonnet-latest'
)

In [ ]:
# Create Pydantic AI runner
from toyaikit.chat.runners import PydanticAIRunner

pydantic_runner = PydanticAIRunner(
    chat_interface=chat_interface,
    agent=pydantic_agent
)

In [ ]:
# Run the Pydantic AI agent (type 'stop' to end)
await pydantic_runner.run()

## 11. MCP (Model Context Protocol)

MCP allows agents to communicate with external tool servers via a standardized protocol.

```
agent <-> MCP server <-> tool
```

### How MCP Works (JSON-RPC)

MCP uses JSON-RPC 2.0 for communication. The handshake sequence:

**1. Initialize request:**
```json
{"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2024-11-05", "capabilities": {"roots": {"listChanged": true}, "sampling": {}}, "clientInfo": {"name": "test-client", "version": "1.0.0"}}}
```

**2. Confirm initialization:**
```json
{"jsonrpc": "2.0", "method": "notifications/initialized"}
```

**3. List available tools:**
```json
{"jsonrpc": "2.0", "id": 2, "method": "tools/list"}
```

**4. Call a tool:**
```json
{"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "search", "arguments": {"query": "how do I run kafka?"}}}
```

The `toyaikit.mcp.MCPClient` handles all of this for us.

In [ ]:
from toyaikit.mcp import MCPClient, SubprocessMCPTransport

In [ ]:
# Start MCP server
command = "uv run python main.py".split()
workdir = "mcp_server"

mcp_client = MCPClient(
    transport=SubprocessMCPTransport(
        server_command=command,
        workdir=workdir
    )
)

In [ ]:
# Start and initialize the MCP server
mcp_client.start_server()
mcp_client.initialize()
mcp_client.initialized()
print("MCP server initialized successfully!")

In [ ]:
# List available tools from MCP server
mcp_client.get_tools()

In [ ]:
# Call a tool directly
result = mcp_client.call_tool('search', {'query': 'how do I run docker?'})
print(result)

In [ ]:
# Use MCP tools with OpenAI Responses runner
from toyaikit.mcp import MCPTools

mcp_tools = MCPTools(mcp_client)

mcp_runner = OpenAIResponsesRunner(
    tools=mcp_tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-4o-mini')
)

In [ ]:
# Run with MCP tools (type 'stop' to end)
mcp_runner.run();

### MCP with Pydantic AI via SSE

You can also connect to an MCP server via Server-Sent Events (SSE):

In [ ]:
from pydantic_ai.mcp import MCPServerSSE

# Connect to MCP server via SSE (requires server running on localhost:8000)
faq_mcp_sse = MCPServerSSE(
    url='http://localhost:8000/sse'
)

mcp_agent = Agent(
    name="faq_agent",
    instructions=developer_prompt,
    model='openai:gpt-4o-mini',  # or 'anthropic:claude-3-7-sonnet-latest'
    toolsets=[faq_mcp_sse]
)

In [ ]:
mcp_pydantic_runner = PydanticAIRunner(
    chat_interface=chat_interface,
    agent=mcp_agent
)

await mcp_pydantic_runner.run();